# 1.Setup

In [36]:
from ri_chat.modeling.utils.chroma_collection import get_chroma_collection
from ri_chat.modeling.utils.embeddings import generate_embeddings, load_embedding_model

import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import pipeline

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import MessagesPlaceholder

import re

In [2]:
# carregando o modelo e os dados
collection = get_chroma_collection()
embedding_model = load_embedding_model()

print(f"Coleção carregada com {collection.count()} chunks.")

2026-11-17 14:38:34.362 | INFO     | ri_chat.modeling.utils.embeddings:load_embedding_model:18 - Loading embedding model 'BAAI/bge-m3' on 'cuda'...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Coleção carregada com 6831 chunks.


# 2.Experimentação

## 2.1.R(etrieval)

In [3]:
def retrieve_pages(
    query: str,
    n_chunks: int = 4,
    periodo: str | None = None,
) -> list[dict]:
    """Busca os chunks mais relevantes e reconstrói as páginas completas."""

    # gera embeddings do prompt do usuario
    query_emb = generate_embeddings(query, embedding_model, show_progress=False)

    # monta um filtro opcional
    where_filter = {"periodo": periodo} if periodo else None

    # busca vetorial pelos chunks mais relevantes
    results = collection.query(
        query_embeddings=[query_emb.tolist()],
        n_results=n_chunks,
        where=where_filter,
    )

    # descobre as paginas unicas encontradas (mantendo a ordem de relevancia)
    seen_pages = []
    for meta in results["metadatas"][0]:
        page_key = (meta["periodo"], meta["tipo_doc"], meta["page_number"])
        if page_key not in seen_pages:
            seen_pages.append(page_key)

    # ordena paginas unicas e buscas todos seus chunks dentro do chroma
    pages = []
    for p_periodo, p_tipo_doc, p_page_num in seen_pages:
        page_data = collection.get(
            where={
                "$and": [
                    {"periodo": p_periodo},
                    {"tipo_doc": p_tipo_doc},
                    {"page_number": p_page_num},
                ]
            }
        )

        # ordena os chunks para reconstruir a pagina na ordem correta
        chunks_sorted = sorted(
            zip(page_data["metadatas"], page_data["documents"]),
            key=lambda item: item[0]["chunk_index"],
        )

        # remove os cabecalhos a partir do segundo chunk
        header = f"[Relatório Itaú Unibanco | Período: {p_periodo} | Demonstrações: {p_tipo_doc} | Página: {p_page_num}]"
        clean_parts = [
            doc.replace(f"{header}\n", "") for _, doc in chunks_sorted
        ]

        pages.append(
            {
                "periodo": p_periodo,
                "tipo_doc": p_tipo_doc,
                "page_number": p_page_num,
                "content": f"{header}\n" + "\n".join(clean_parts),
            }
        )

    return pages




In [4]:
# testando
paginas = retrieve_pages(
    query="Qual foi o lucro líquido recorrente e o ROE em 4T2024?",
    n_chunks=3,
    periodo="4T2024",
)
print(f"Páginas únicas recuperadas: {len(paginas)}\n")
for p in paginas:
    print(f"=== {p['periodo']} ({p['tipo_doc']}) - Página {p['page_number']} ===")
    print(p["content"][:], "...\n")

Páginas únicas recuperadas: 2

=== 4T2024 (completas) - Página 8 ===
[Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 8]
###### Análise do desempenho do 4o trimestre de 2024 

###### **Comentários da Administração**
Nosso resultado recorrente gerencial alcançou R$ 10,9 bilhões no quarto trimestre de 2024, com aumento trimestral de 2,0%. O retorno recorrente gerencial sobre o patrimônio líquido foi de 22,1% no consolidado e de 23,4% nas operações no Brasil. A carteira de crédito avançou 6,3% no consolidado e 5,8% no Brasil no trimestre. Na carteira PF, os destaques foram os crescimentos de 5,6% do crédito imobiliário, de 1,8% da carteira de veículos e de 6,8% de cartão de crédito, que é sazonalmente maior no final do ano. Em cartão de crédito, o crescimento da carteira foi de 10,9% no trimestre nos segmentos Personnalité e Uniclass (crescimento de 17,5% em relação ao ano anterior). O crescimento da carteira de micro, pequenas e médias empresas foi de 8,1% 

Perfeito resultado, nossa busca está funcionando corretamente.

## 2.2.Quantização

Iremos usar o modelo Qwen com 7 bilhões de parâmetros.

Padrão desse modelo é que cada parâmetro seja armanezado com uma precisão de 16 bits (float16), dessa forma, cada parâmetro ocupa na memória um espaço de 2 bytes, logo: 7 bilhões X 2 bytes = 14 GB de VRAM serão necessários! Esse é um modelo extremamente econômico. Eu até tenho em minha disposição, o ambiente do Colab Pro, com GPU's na casa dos 80 GB de VRAM, entretanto, para deixar o desafio mais interessante, rodarei esse projeto em minha máquina local.

E por isso, a quantização será peça fundamental desse processo.

In [5]:
# meu setup
props = torch.cuda.get_device_properties(0)
free_vram, total_vram = torch.cuda.mem_get_info(0)

print('Meu setup local:')
print(f"GPU: {props.name} | Total: {total_vram / 1024**3:.1f} GB | Livre: {free_vram / 1024**3:.1f} GB")

Meu setup local:
GPU: NVIDIA GeForce RTX 3060 | Total: 12.0 GB | Livre: 8.8 GB


In [6]:
# modelo escolhido
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [7]:
# configurando o bit and bytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # diz ao transformers para comprimir os pesos para 4 bits ao carregar o modelo na GPU
    bnb_4bit_quant_type="nf4",  # tecnica de normalizacao dos pesos
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # os pesos sao armezados em 4bits mas na hora das operacoes sao convertidos em 16 bits
)


In [8]:
# # Carregando o modelo
# tokenizados
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# carregando o modelo quantizado
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='cuda',
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [9]:
props = torch.cuda.get_device_properties(0)
free_vram, total_vram = torch.cuda.mem_get_info(0)

print('Meu setup local:')
print(f"GPU: {props.name} | Total: {total_vram / 1024**3:.1f} GB | Livre: {free_vram / 1024**3:.1f} GB")

Meu setup local:
GPU: NVIDIA GeForce RTX 3060 | Total: 12.0 GB | Livre: 3.5 GB


## 2.3.LangChain

In [10]:
# criando a pipeline de geracao de texto
txt_gen_pipe = pipeline(
    task='text-generation',
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,

)

#empacotando tudo no chat do langchain
hf_pipeline = HuggingFacePipeline(pipeline=txt_gen_pipe)
chat_model = ChatHuggingFace(llm=hf_pipeline)

In [11]:
txt_gen_pipe

TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}

In [12]:
hf_pipeline

HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain-huggingface': '1.2.3'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen2.5-7B-Instruct')

In [13]:
chat_model

ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain-huggingface': '1.2.3'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain-huggingface': '1.2.3'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen2.5-7B-Instruct'), tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-7B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=Fa

In [14]:
# testando
messages = [
    ("system", "Você é um analista financeiro sênior especialista em bancos brasileiros. Responda de forma direta."),
    ("human", "Em uma frase, o que é o ROE (Return on Equity) de um banco?"),
]
response = chat_model.invoke(messages)
print(response.content)

O ROE (Return on Equity) de um banco é a taxa de retorno sobre o capital próprio investido na empresa, indicando sua eficiência no uso do capital próprio para gerar lucros.


In [15]:
print(response)

content='O ROE (Return on Equity) de um banco é a taxa de retorno sobre o capital próprio investido na empresa, indicando sua eficiência no uso do capital próprio para gerar lucros.' additional_kwargs={} response_metadata={} id='lc_run--01a1ea97-2e64-76c6-8ed2-e32987da81a2-0' tool_calls=[] invalid_tool_calls=[]


In [16]:
SYSTEM_PROMPT = """Você é um assistente especialista em Relações com Investidores (RI) do Itaú Unibanco.
Sua tarefa é responder às perguntas do usuário utilizando EXCLUSIVAMENTE as informações contidas no CONTEXTO abaixo, extraído dos relatórios financeiros oficiais.

Regras obrigatórias:
1. Baseie-se apenas nos dados do contexto. Se a resposta não estiver no contexto, diga claramente que a informação não consta nos trechos recuperados. Nunca invente números.
2. Cite sempre a fonte de cada informação utilizando os cabeçalhos presentes no contexto (informando o Período, o Tipo de Demonstração e o Número da Página).
3. Mantenha um tom técnico, objetivo e imparcial, em conformidade com as diretrizes da CVM (nunca faça recomendações de compra ou venda de ativos)."""


In [17]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "CONTEXTO:\n{context}\n\nPERGUNTA:\n{question}"),
    ]
)

# Encadeando o fluxo com LCEL (Prompt -> LLM -> String)
rag_chain = prompt_template | chat_model | StrOutputParser()

In [18]:
# teste rapido
q = "Qual foi o lucro líquido recorrente (resultado recorrente gerencial) e o ROE no 4T2024?"
pages = retrieve_pages(
    query=q,
    n_chunks=3,
    periodo="4T2024",
)

# conteudo separado por linhas divisorias
context_text = "\n\n---\n\n".join(page["content"] for page in pages)

# invoca a chain do langchain e passa contexto e pergunta
answer = rag_chain.invoke(
    {
        "context": context_text,
        "question": q,
    }
)


In [19]:
print(context_text)

[Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 8]
###### Análise do desempenho do 4o trimestre de 2024 

###### **Comentários da Administração**
Nosso resultado recorrente gerencial alcançou R$ 10,9 bilhões no quarto trimestre de 2024, com aumento trimestral de 2,0%. O retorno recorrente gerencial sobre o patrimônio líquido foi de 22,1% no consolidado e de 23,4% nas operações no Brasil. A carteira de crédito avançou 6,3% no consolidado e 5,8% no Brasil no trimestre. Na carteira PF, os destaques foram os crescimentos de 5,6% do crédito imobiliário, de 1,8% da carteira de veículos e de 6,8% de cartão de crédito, que é sazonalmente maior no final do ano. Em cartão de crédito, o crescimento da carteira foi de 10,9% no trimestre nos segmentos Personnalité e Uniclass (crescimento de 17,5% em relação ao ano anterior). O crescimento da carteira de micro, pequenas e médias empresas foi de 8,1% no trimestre, e é explicado pela variação cambial do período e pela o

In [20]:
print(answer)

O lucro líquido recorrente (resultado recorrente gerencial) no 4T2024 foi de R$ 10,9 bilhões, com um aumento de 2,0% em comparação ao 3T2024. O retorno recorrente gerencial sobre o patrimônio líquido foi de 22,1% no 4T2024, representando uma redução de 0,6 p.p. em relação ao 3T2024. Essas informações estão disponíveis na seção "Análise do desempenho do 4o trimestre de 2024" do relatório do Itaú Unibanco.


In [21]:
# criando uma funcao
def ask_ri_chat(
    question: str,
    periodo: str | None = None,
    n_chunks: int = 3,
) -> str:
    """Executa o fluxo RAG completo: busca páginas no ChromaDB e gera resposta com o LLM."""

    # recupera as paginas compeltas do chromaDB
    pages = retrieve_pages(
        query=question,
        n_chunks=n_chunks,
        periodo=periodo,
    )

    # conteudo separado por linhas divisorias
    context_text = "\n\n---\n\n".join(page["content"] for page in pages)

    # invoca a chain do langchain e passa contexto e pergunta
    answer = rag_chain.invoke(
        {
            "context": context_text,
            "question": question,
        }
    )

    return answer


In [22]:
# teste 1
resposta = ask_ri_chat(
    question="Qual foi o lucro líquido recorrente (resultado recorrente gerencial) e o ROE no 4T2024?",
    periodo="4T2024",
    n_chunks=3,
)
print(resposta)

O lucro líquido recorrente (resultado recorrente gerencial) no 4T2024 foi de R$ 10,9 bilhões, com um aumento de 2,0% em comparação ao 3T2024. O retorno recorrente gerencial sobre o patrimônio líquido foi de 22,1% no 4T2024, representando uma redução de 0,6 p.p. em relação ao 3T2024. Essas informações estão disponíveis na seção "Análise do desempenho do 4o trimestre de 2024" do relatório do Itaú Unibanco.


In [23]:
# teste 2
resposta = ask_ri_chat(
    question="Qual foi o lucro líquido recorrente (resultado recorrente gerencial) e o ROE no quarto trimestre de 2024?",
    periodo="4T2024",
    n_chunks=3,
)

print(resposta)

O lucro líquido recorrente (resultado recorrente gerencial) no quarto trimestre de 2024 foi de R$ 10,9 bilhões, com um aumento de 2,0% em comparação ao trimestre anterior.

O retorno recorrente gerencial sobre o patrimônio líquido (ROE) no quarto trimestre de 2024 foi de 22,1%.

Fonte: [Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 8]


Perguntas levemente diferentes, mas ambas com a mesma resposta. Perfeito!

In [32]:
# agora, quero testar uma abordagem para extrair o perido ao qual o usuario esta se referindo em sua pergunta
# vamos ao teste
PERIODO_PROMPT_TEST = """Você é um classificador temporal financeiro.
Sua única tarefa é identificar a qual trimestre fiscal (no formato XT20YY, onde X é 1, 2, 3 ou 4 e 20YY é o ano com 4 dígitos) a pergunta do usuário se refere.

Regras de equivalência de meses:
- 1T: janeiro, fevereiro, março (meses 1, 2, 3 / três primeiros meses)
- 2T: abril, maio, junho (meses 4, 5, 6 / segundo trimestre / fechamento do 1º semestre)
- 3T: julho, agosto, setembro (meses 7, 8, 9)
- 4T: outubro, novembro, dezembro (meses 10, 11, 12 / quarto trimestre / últimos 3 meses do ano)

Responda EXCLUSIVAMENTE com o código no formato XT20YY (ex: 1T2024, 4T2024, 2T2025) ou com a palavra NONE caso a pergunta não mencione nenhum período específico. Não escreva mais nenhuma palavra."""
chat_prompt_template = ChatPromptTemplate.from_messages([
            ("system", PERIODO_PROMPT_TEST),
            ("human", "{question}"),
        ])
period_chain = chat_prompt_template | chat_model | StrOutputParser()



def extract_period_llm(question: str) -> str | None:
    """Usa o LLM para interpretar qualquer menção temporal e converter em 'XT20YY'."""
    result = period_chain.invoke({"question": question}).strip()
    return None if result == "NONE" else result

In [33]:
testes = [
    "Qual o ROE no 4T2024?",
    "Qual foi o lucro no T424?",
    "Como foi o quarto trimestre de dois mil e vinte e quatro?",
    "Qual foi o resultado nos 3 primeiros meses de 2024?",
    "Quanto foi o lucro em jan, fev e mar de 2024?",
    "Mostre os dados de janeiro, fevereiro e março de 2024",
    "Como foram os meses 1, 2 e 3 de 2024?",
    "Qual o desempenho acumulado em 01/2024, 02/2024 e 03/2024?",
    "Qual foi o lucro no trimestre 4 de 2024?",
    "O que é o índice de Basileia?",
    "O que é ROI?"
]

for pergunta in testes:
    periodo_detectado = extract_period_llm(pergunta)
    print(f"{pergunta:62} -> {periodo_detectado}")

Qual o ROE no 4T2024?                                          -> 4T2024
Qual foi o lucro no T424?                                      -> 4T2024
Como foi o quarto trimestre de dois mil e vinte e quatro?      -> 4T2024
Qual foi o resultado nos 3 primeiros meses de 2024?            -> 1T2024
Quanto foi o lucro em jan, fev e mar de 2024?                  -> 1T2024
Mostre os dados de janeiro, fevereiro e março de 2024          -> 1T2024
Como foram os meses 1, 2 e 3 de 2024?                          -> 1T2024
Qual o desempenho acumulado em 01/2024, 02/2024 e 03/2024?     -> 1T2024
Qual foi o lucro no trimestre 4 de 2024?                       -> 4T2024
O que é o índice de Basileia?                                  -> None
O que é ROI?                                                   -> None


## 2.4.Memória do Chat e LangChain

In [41]:
# prompt extrator de periodo
PERIODO_PROMPT_MEM = """Você é um classificador temporal financeiro.
Identifique a qual trimestre fiscal (no formato XT20YY, ex: 1T2024, 4T2024, 2T2025) a pergunta atual do usuário se refere.
Se a pergunta atual fizer referência à conversa anterior (ex: "e nesse mesmo período?", "e o lucro?"), use o período mencionado no histórico da conversa.

Regras de equivalência:
- Abreviações curtas ou invertidas: 4T24, T424, 4T/24 -> 4T2024
- 1T: janeiro, fevereiro, março (meses 1, 2, 3 / três primeiros meses)
- 2T: abril, maio, junho (meses 4, 5, 6 / segundo trimestre / 1º semestre)
- 3T: julho, agosto, setembro (meses 7, 8, 9)
- 4T: outubro, novembro, dezembro (meses 10, 11, 12 / quarto trimestre)

Responda EXCLUSIVAMENTE com o código XT20YY ou NONE. Não escreva mais nenhuma palavra."""

chat_prompt_template = ChatPromptTemplate.from_messages([
        ("system", PERIODO_PROMPT_MEM),
        ("human", "HISTÓRICO DA CONVERSA:\n{history_text}\n\nPERGUNTA ATUAL:\n{question}"),
    ])
period_chain_mem = chat_prompt_template | chat_model | StrOutputParser()


# chain principal do rag
chat_prompt_template = ChatPromptTemplate.from_messages([
            ("system", SYSTEM_PROMPT),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "CONTEXTO:\n{context}\n\nPERGUNTA:\n{question}"),
        ])
rag_chain_mem = chat_prompt_template | chat_model | StrOutputParser()

In [42]:
# Lista que guarda a memória da sessão de conversa
chat_history = []


def chat_with_ri(question: str, n_chunks: int = 3) -> str:
    """Conversa com o assistente RI mantendo memória e detectando o período automaticamente."""

    # Transforma o histórico em texto apenas para o classificador ler sem se confundir
    history_text = "\n".join(f"{msg.type}: {msg.content}" for msg in chat_history) or "Sem histórico."

    # 1. Descobre o período olhando para o histórico em texto + pergunta atual
    raw_period = period_chain_mem.invoke(
        {"question": question, "history_text": history_text}
    ).strip()

    periodo = raw_period if re.match(r"^[1-4]T202[4-6]$", raw_period) else None
    print(f"[Debug] Período detectado automaticamente: {periodo}")

    # 2. Busca as páginas completas no ChromaDB
    pages = retrieve_pages(query=question, n_chunks=n_chunks, periodo=periodo)
    context_text = "\n\n---\n\n".join(page["content"] for page in pages)

    # 3. Gera a resposta passando o histórico (MessagesPlaceholder) + contexto + pergunta
    answer = rag_chain_mem.invoke(
        {
            "chat_history": chat_history,
            "context": context_text,
            "question": question,
        }
    )

    # 4. Salva APENAS a pergunta e a resposta limpa na memória
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer

In [52]:
# Limpa a memória antes de começar o teste
chat_history.clear()

print("--- TURNO 1 ---")
q = "Qual foi o resultado recorrente gerencial e o ROE nos 3 últimos meses de 2024?"
resp = chat_with_ri(q)
print(f'[Humano]: {q}')
print(f'[IA]: {resp}')
print("="*100)


print("\n--- TURNO 2 (Usando a memória!) ---")
q = "E de quanto foi o crescimento da carteira de crédito?"
resp = chat_with_ri(q)
print(f'[Humano]: {q}')
print(f'[IA]: {resp}')
print("="*100)


print("\n--- TURNO 3 (Usando a memória!) ---")
q = "E identificou algum risco de prejuízo dígno de nota?"
resp = chat_with_ri(q)
print(f'[Humano]: {q}')
print(f'[IA]: {resp}')
print("="*100)


print("\n--- TURNO 4 (Usando a memória!) ---")
q = "Me passe uma deliciosa receita de bolo!"
resp = chat_with_ri(q)
print(f'[Humano]: {q}')
print(f'[IA]: {resp}')
print("="*100)


print("\n--- TURNO 5 (Usando a memória!) ---")
q = "Certo, obrigado pelo suporte!"
resp = chat_with_ri(q)
print(f'[Humano]: {q}')
print(f'[IA]: {resp}')
print("="*100)


print("\n--- TURNO 6 (Usando a memória!) ---")
q = "Ah, acabei esquecendo, qual foi o ROE mesmo?"
resp = chat_with_ri(q)
print(f'[Humano]: {q}')
print(f'[IA]: {resp}')
print("="*100)

--- TURNO 1 ---
[Debug] Período detectado automaticamente: 4T2024
[Humano]: Qual foi o resultado recorrente gerencial e o ROE nos 3 últimos meses de 2024?
[IA]: O resultado recorrente gerencial no 4o trimestre de 2024 foi de R$ 10,9 bilhões, representando um aumento de 2,0% em comparação com o mesmo período do ano anterior (4T2023).

O retorno recorrente gerencial sobre o patrimônio líquido (ROE) no 4o trimestre de 2024 foi de 22,1% no consolidado e de 23,4% nas operações no Brasil.

Fonte: [Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 8]

--- TURNO 2 (Usando a memória!) ---
[Debug] Período detectado automaticamente: 4T2024
[Humano]: E de quanto foi o crescimento da carteira de crédito?
[IA]: A carteira de crédito no 4o trimestre de 2024 cresceu 6,3% em comparação com o mesmo período do ano anterior (4T2023).

Fonte: [Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 22]

--- TURNO 3 (Usando a memória!) ---
[Debug] Período 

Pontos positivos encontrados nos testes:
* Ao falar receita sobre receita de bolo, o modelo foi capaz de dar uma resposta negativa.
* Ao pedir sobre riscos, o modelo não inventou informações.
* Mesmo após duas iterações sem mencionar o período, o modelo foi capaz de recuperar essa informação do histórico.